**INVENTORY AND PROFITABILITY ANALYSIS**

This project analyzes project-level financial performance for a carpentry business using SQLite and Python.

The objective is to:
- Evaluate profitability across projects
- Identify high-margin and low-margin projects
- Detect loss-making projects
- Analyze performance by product type and client

Load Data

In [ ]:
import pandas as pd

df = pd.read_csv('projects.csv')
df.head()

,PROJECT_ID,PROJECT_NAME,CLIENT,PRODUCT_TYPE,QUANTITY,COST_TOTAL,PRICE,PROFIT,MARGIN
0,P001,11 Entrepaños,Client 007,SHELVES,11,3095.0,6949.44,3854.44,0.554640
1,P002,3 Mesa redonda,Client 002,TABLE,3,7770.0,14426.67,6656.67,0.461414
2,P003,Bufetera y repisas,Client 013,FURNITURE,1,10653.0,8750.00,-1903.00,-0.217486
3,P004,Cabecera/closet,Client 014,CLOSET,1,25680.0,55319.00,29639.00,0.535783
4,P005,Cajonera,Client 010,FURNITURE,1,7655.0,23130.00,15475.00,0.669045


Create SQLite Database

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('projects', conn, index=False, if_exists='replace')

30

Create Analysis View

In [ ]:
query = """
CREATE VIEW project_analysis AS
SELECT
    PROJECT_ID,
    PROJECT_NAME,
    CLIENT,
    PRODUCT_TYPE,
    QUANTITY,
    COST_TOTAL,
    PRICE,
    PROFIT,
    MARGIN
FROM projects;
"""

conn.execute(query)

Most Profitable Projects

In [ ]:
query = """
SELECT PROJECT_NAME, PROFIT
FROM project_analysis
ORDER BY PROFIT DESC;
"""

pd.read_sql(query, conn)

,PROJECT_NAME,PROFIT
0,Cabecera/closet,29639.00
1,Puerta en Nogal,21930.00
2,Cajonera,20190.00
3,Cajonera,18375.00
4,Puerta en Parota,17960.59
5,Puerta en Encino,16017.06
6,Librero Grande,15850.00
7,Cajonera,15475.00
8,Puerta en Pino Estufado,13830.00
9,Closet Completo,11520.50


### Insight
The results show which projects generate the highest profit, indicating strong pricing strategies and efficient cost management.

Loss-Making Projects

In [ ]:
query = """
SELECT PROJECT_NAME, PROFIT
FROM project_analysis
WHERE PROFIT < 0;
"""

pd.read_sql(query, conn)

,PROJECT_NAME,PROFIT
0,Bufetera y repisas,-1903.00
1,Linea Cubrica,-1690.79


### Insight
Some projects show negative profit, suggesting pricing or cost inefficiencies that should be reviewed.

Profitability by Product Type

In [ ]:
query = """
SELECT
    PRODUCT_TYPE,
    AVG(MARGIN) AS AVG_MARGIN
FROM project_analysis
GROUP BY PRODUCT_TYPE
ORDER BY AVG_MARGIN DESC;
"""

pd.read_sql(query, conn)

,PRODUCT_TYPE,AVG_MARGIN
0,RENEWAL,0.548364
1,SHELVES,0.526740
2,CABINET DOORS,0.523013
3,DESK,0.521875
4,DOORS,0.494297
5,CLOSET,0.440112
6,TABLE,0.432902
7,FURNITURE,0.396160
8,DOOR,0.384483
9,BOOKSELLER,0.265828


### Insight
Different product types show variability in margins, highlighting which categories are more profitable.

Revenue by Client

In [ ]:
query = """
SELECT
    CLIENT,
    SUM(PRICE) AS TOTAL_REVENUE
FROM project_analysis
GROUP BY CLIENT
ORDER BY TOTAL_REVENUE DESC;
"""

pd.read_sql(query, conn)

,CLIENT,TOTAL_REVENUE
0,Client 001,153077.65
1,Client 002,86358.90
2,Client 014,55319.00
3,Client 016,53322.75
4,Client 013,35640.50
5,Client 007,27168.33
6,Client 012,23738.31
7,Client 015,23130.00
8,Client 010,23130.00
9,Client 009,10906.45


### Insight
This analysis identifies key clients contributing most to revenue.

Conclusion

Load Materials

In [ ]:
materials = pd.read_csv('materials.csv')
materials.head()

,PROJECT_NAME,CLIENT,MATERIAL,QUANTITY,UNIT_COST,TOTAL_COST,CALCULATED_TOTAL,COST_DIFFERENCE,CATEGORY,TYPE,GROUP_NAME,DIMENSIONS,THICKNESS,WIDTH,LENGTH,VOLUME,PRODUCT_TYPE,PROJECT_ID
0,Puerta en Parota,Client 001,TABLA 1X6X8 PAROTA,8.0,720.0,5760.0,5760.0,0.0,TABLAS,BOARD,RAW_MATERIAL,1X6X8,1.0,6.0,8.0,48.0,DOOR,P024
1,Puerta en Parota,Client 001,TABLA 1X8X8 PAROTA,20.0,650.0,13000.0,13000.0,0.0,TABLAS,BOARD,RAW_MATERIAL,1X8X8,1.0,8.0,8.0,64.0,DOOR,P024
2,Puerta en Parota,Client 001,BARNIZ EXTERIOR,2.0,350.0,700.0,700.0,0.0,SOLVENTE,VARNISH,FINISHING,NaN,NaN,NaN,NaN,NaN,DOOR,P024
3,Puerta en Parota,Client 001,FONDO SELLADOR,2.0,300.0,600.0,600.0,0.0,SOLVENTE,SEALER,FINISHING,NaN,NaN,NaN,NaN,NaN,DOOR,P024
4,Puerta en Parota,Client 001,LIJAS,20.0,20.0,400.0,400.0,0.0,LIJAS,SANDPAPER,FINISHING,NaN,NaN,NaN,NaN,NaN,DOOR,P024


In [ ]:
materials.to_sql('materials', conn, index=False, if_exists='replace')

429

Most Expensive Materials

In [ ]:
query = """
SELECT
    MATERIAL,
    SUM(TOTAL_COST) AS TOTAL_COST
FROM materials
GROUP BY MATERIAL
ORDER BY TOTAL_COST DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,MATERIAL,TOTAL_COST
0,TRIPLAY 15MM,31050.0
1,16MM,23000.0
2,TABLA 1X8X8 NOGAL AMERICANO,22000.0
3,TRIPLAY 16MM,13400.0
4,TABLA 1X8X8 PAROTA,13000.0
5,FLETE,12150.0
6,TABLA 1X8X8 ENCINO,11000.0
7,MDF 16MM,7675.0
8,TRIPLAY 6MM,7650.0
9,6MM,7570.0


The analysis identifies the main cost drivers, showing which materials contribute most to total project costs.

Cost distribution by category

In [ ]:
query = """
SELECT
    CATEGORY,
    SUM(TOTAL_COST) AS CATEGORY_COST
FROM materials
GROUP BY CATEGORY
ORDER BY CATEGORY_COST DESC;
"""

pd.read_sql(query, conn)

,CATEGORY,CATEGORY_COST
0,TABLAS,82990.0
1,ENCHAPADOS,72885.0
2,FERRETERIA,62520.0
3,SOLVENTE,33315.0
4,OTRO,18895.0
5,MDF,13925.0
6,TERMINACION,12672.0
7,MELAMINA,11550.0
8,LIJAS,4750.0
9,TINTA,2610.0


In [ ]:
query = """
SELECT
    p.PROJECT_NAME,
    p.PRODUCT_TYPE,
    p.PROFIT,
    SUM(m.TOTAL_COST) AS MATERIAL_COST
FROM projects p
JOIN materials m
ON p.PROJECT_ID = m.PROJECT_ID
GROUP BY p.PROJECT_ID
ORDER BY p.PROFIT DESC;
"""

pd.read_sql(query, conn)

,PROJECT_NAME,PRODUCT_TYPE,PROFIT,MATERIAL_COST
0,Cabecera/closet,CLOSET,29639.00,25680.0
1,Puerta en Nogal,DOOR,21930.00,31710.0
2,Cajonera,FURNITURE,20190.00,15350.0
3,Cajonera,FURNITURE,18375.00,15350.0
4,Puerta en Parota,DOOR,17960.59,22070.0
5,Puerta en Encino,DOOR,16017.06,17350.0
6,Librero Grande,BOOKSELLER,15850.00,43775.0
7,Cajonera,FURNITURE,15475.00,15350.0
8,Puerta en Pino Estufado,DOOR,13830.00,12210.0
9,Closet Completo,CLOSET,11520.50,15370.0


In [ ]:
query = """
SELECT
    PROJECT_NAME,
    PRODUCT_TYPE,
    COST_TOTAL / QUANTITY AS COST_PER_UNIT,
    PRICE / QUANTITY AS PRICE_PER_UNIT
FROM projects;
"""

pd.read_sql(query, conn)

,PROJECT_NAME,PRODUCT_TYPE,COST_PER_UNIT,PRICE_PER_UNIT
0,11 Entrepaños,SHELVES,281.363636,631.767273
1,3 Mesa redonda,TABLE,2590.000000,4808.890000
2,Bufetera y repisas,FURNITURE,10653.000000,8750.000000
3,Cabecera/closet,CLOSET,25680.000000,55319.000000
4,Cajonera,FURNITURE,7655.000000,23130.000000
5,Cajonera,FURNITURE,4755.000000,23130.000000
6,Cajonera,FURNITURE,2940.000000,23130.000000
7,Closet Completo,CLOSET,15370.000000,26890.500000
8,Closet lado ventana,CLOSET,19955.000000,30192.750000
9,Closet sencillo 140 ancho x 210 alto y 50 fondo,CLOSET,3815.000000,7027.890000


In [ ]:
query = """
SELECT
    PROJECT_NAME,
    PRODUCT_TYPE,
    COST_TOTAL,
    PRICE,
    PROFIT,
    MARGIN
FROM projects
ORDER BY MARGIN ASC;
"""

pd.read_sql(query, conn)

,PROJECT_NAME,PRODUCT_TYPE,COST_TOTAL,PRICE,PROFIT,MARGIN
0,Bufetera y repisas,FURNITURE,10653.0,8750.00,-1903.00,-0.217486
1,Linea Cubrica,FURNITURE,9725.0,8034.21,-1690.79,-0.210449
2,Puerta de tambor,DOOR,3670.0,3878.56,208.56,0.053773
3,Librero Grande,BOOKSELLER,43775.0,59625.00,15850.00,0.265828
4,Mesa de trabajo,TABLE,4935.0,6748.89,1813.89,0.268769
5,Closet lado ventana,CLOSET,19955.0,30192.75,10237.75,0.339080
6,Vanity,FURNITURE,5840.0,9263.00,3423.00,0.369535
7,Linea Normal,FURNITURE,5260.0,8617.26,3357.26,0.389597
8,Puerta en Nogal,DOOR,31710.0,53640.00,21930.00,0.408837
9,Closet Completo,CLOSET,15370.0,26890.50,11520.50,0.428423


In [ ]:
query = """
SELECT
    p.PRODUCT_TYPE,
    AVG(m.TOTAL_COST) AS AVG_MATERIAL_COST
FROM materials m
JOIN projects p
ON m.PROJECT_ID = p.PROJECT_ID
GROUP BY p.PRODUCT_TYPE;
"""

pd.read_sql(query, conn)

,PRODUCT_TYPE,AVG_MATERIAL_COST
0,BOOKSELLER,2431.944444
1,CABINET DOORS,212.391304
2,CLOSET,1012.812500
3,DESK,327.857143
4,DOOR,1450.166667
5,DOORS,383.571429
6,FURNITURE,490.165680
7,RENEWAL,242.142857
8,SHELVES,263.928571
9,TABLE,403.564103


### Key Findings

- Material costs are concentrated in a small number of high-impact items
- Profitability varies significantly across projects and product types
- Some projects operate with low or negative margins, indicating inefficiencies
- Unit-level analysis reveals differences in pricing strategy
- Material consumption plays a critical role in overall cost structure

### Business Impact

These insights can support:
- Cost reduction strategies
- Pricing optimization
- Focus on high-margin products
- Better resource allocation

### Key Findings

- Profitability varies across projects and product types
- Some projects operate at a loss, indicating inefficiencies
- Certain product categories generate higher margins
- Revenue is concentrated among specific clients

### Business Impact

These insights can support:
- Pricing optimization
- Cost control strategies
- Focus on high-margin products
- Better client targeting